In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from scenarios.price_scenarios import build_zonal_price_matrix, zone_annual_cf, CONTRACT_START

own_zone = "nord"
reference_zone = "sicilia"

price_own_zone = build_zonal_price_matrix(own_zone)
price_reference_zone = build_zonal_price_matrix(reference_zone)
price_own_zone.head()

price_scenario Delayed transition                                      \
weather_year                 1991        1992        1993        1994   
contract_year                                                           
2026                   103.875909  104.316244  104.055878  104.264543   
2027                   101.454880  101.895215  101.634849  101.843513   
2028                    99.033851   99.474185   99.213819   99.422484   
2029                    96.612821   97.053156   96.792790   97.001455   
2030                    94.191792   94.632127   94.371761   94.580426   

price_scenario                                                              \
weather_year          1995        1996        1997        1998        1999   
contract_year                                                                
2026            103.783605  104.723265  103.665408  103.587186  104.067303   
2027            101.362575  102.302236  101.244378  101.166157  101.646274   
2028             98.941546   99.881206   98.823349   98.745128   99.225244   
2029             96.520517   97.460177   96.402320   96.324099   96.804215   
2030             94.099488   95.039148   93.981291   93.903069   94.383186   

price_scenario              ... Net Zero 2050                          \
weather_year          2000  ...          2014        2015        2016   
contract_year               ...                                         
2026            104.040144  ...    110.643072  109.916631  110.109090   
2027            101.619114  ...    111.375168  110.648728  110.841187   
2028             99.198085  ...    112.107264  111.380824  111.573283   
2029             96.777056  ...    112.839361  112.112920  112.305379   
2030             94.356027  ...    113.571457  112.845017  113.037476   

price_scenario                                                              \
weather_year          2017        2018        2019        2020        2021   
contract_year                                                                
2026            109.508766  110.246683  109.840647  109.721979  109.802884   
2027            110.240862  110.978779  110.572743  110.454075  110.534981   
2028            110.972959  111.710875  111.304839  111.186172  111.267077   
2029            111.705055  112.442972  112.036936  111.918268  111.999173   
2030            112.437151  113.175068  112.769032  112.650364  112.731270   

price_scenario                          
weather_year          2022        2023  
contract_year                           
2026            109.546311  109.692967  
2027            110.278407  110.425063  
2028            111.010504  111.157160  
2029            111.742600  111.889256  
2030            112.474696  112.621352  

[5 rows x 99 columns]

In [2]:
annual_solar_cf = zone_annual_cf("solar", own_zone)
annual_solar_cf.describe()

count    33.000000
mean      0.181007
std       0.006687
min       0.167394
25%       0.176553
50%       0.181215
75%       0.186130
max       0.190684
dtype: float64

In [3]:
STRIKE_PRICE_SOLAR = 56.83
HOURS_PER_YEAR = 8760

pap_payment_per_mw = STRIKE_PRICE_SOLAR * annual_solar_cf * HOURS_PER_YEAR
pap_payment_per_mw.describe()

count       33.000000
mean     90110.949264
std       3329.118366
min      83333.888435
25%      87893.422921
50%      90214.482238
75%      92661.070039
max      94928.422168
dtype: float64

In [4]:
from scenarios.load_profile import load_profile_for_archetype

annual_kwh = 20_000_000
load = load_profile_for_archetype("chemicals", annual_kwh)
load.sum() / 1000

np.float64(20000.0)

In [5]:
annual_load_mwh = load.sum() / 1000
CONTRACTED_MW = 5

residual_mwh = (annual_load_mwh - annual_solar_cf * HOURS_PER_YEAR * CONTRACTED_MW).clip(lower=0)
residual_mwh.describe()

count       33.000000
mean     12071.885513
std        292.901493
min      11648.036058
25%      11847.521552
50%      12062.776506
75%      12266.987250
max      12668.142844
dtype: float64

In [6]:
payment_total = pap_payment_per_mw * CONTRACTED_MW

columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_price = price_own_zone[(scenario, weather_year)]
    columns[(scenario, weather_year)] = payment_total[weather_year] + residual_mwh[weather_year] * spot_price

pap_solar_cost_matrix = pd.DataFrame(columns)
pap_solar_cost_matrix.columns.names = ["price_scenario", "weather_year"]
pap_solar_cost_matrix.index.name = "contract_year"
pap_solar_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.697635e+06  1.721049e+06  1.707469e+06  1.718479e+06   
2027                 1.668764e+06  1.691252e+06  1.678203e+06  1.688780e+06   
2028                 1.639892e+06  1.661454e+06  1.648938e+06  1.659081e+06   
2029                 1.611021e+06  1.631657e+06  1.619672e+06  1.629383e+06   
2030                 1.582149e+06  1.601859e+06  1.590407e+06  1.599684e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.692884e+06  1.743319e+06  1.686948e+06  1.682637e+06   
2027            1.664201e+06  1.712649e+06  1.658499e+06  1.654364e+06   
2028            1.635517e+06  1.681979e+06  1.630051e+06  1.626091e+06   
2029            1.606834e+06  1.651309e+06  1.601602e+06  1.597818e+06   
2030            1.578151e+06  1.620639e+06  1.573153e+06  1.569544e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.708147e+06  1.706214e+06  ...  1.815988e+06  1.773613e+06   
2027            1.678854e+06  1.677003e+06  ...  1.825231e+06  1.782398e+06   
2028            1.649561e+06  1.647792e+06  ...  1.834474e+06  1.791182e+06   
2029            1.620267e+06  1.618581e+06  ...  1.843716e+06  1.799967e+06   
2030            1.590974e+06  1.589370e+06  ...  1.852959e+06  1.808752e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.784925e+06  1.750204e+06  1.792926e+06  1.769508e+06   
2027            1.793834e+06  1.758732e+06  1.801921e+06  1.778249e+06   
2028            1.802742e+06  1.767259e+06  1.810917e+06  1.786989e+06   
2029            1.811651e+06  1.775787e+06  1.819912e+06  1.795730e+06   
2030            1.820559e+06  1.784314e+06  1.828907e+06  1.804471e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.762305e+06  1.766904e+06  1.751863e+06  1.760555e+06  
2027            1.770965e+06  1.775615e+06  1.760408e+06  1.769196e+06  
2028            1.779626e+06  1.784326e+06  1.768952e+06  1.777837e+06  
2029            1.788287e+06  1.793037e+06  1.777497e+06  1.786478e+06  
2030            1.796947e+06  1.801748e+06  1.786041e+06  1.795119e+06  

[5 rows x 99 columns]

In [7]:
basis_risk = price_own_zone - price_reference_zone
basis_risk.loc[CONTRACT_START].describe()

count    9.900000e+01
mean     2.153160e-15
std      4.270334e-01
min     -6.005946e-01
25%     -4.115476e-01
50%     -1.651287e-02
75%      1.980173e-01
max      1.023159e+00
Name: 2026, dtype: float64

In [8]:
STRIKE_PRICE_WIND = 72.85
CONTRACTED_MW_VPPA = 5

wind_cf_reference = zone_annual_cf("wind", reference_zone)

vppa_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_own = price_own_zone[(scenario, weather_year)]
    spot_reference = price_reference_zone[(scenario, weather_year)]
    settlement = (STRIKE_PRICE_WIND - spot_reference) * wind_cf_reference[weather_year] * HOURS_PER_YEAR * CONTRACTED_MW_VPPA
    vppa_columns[(scenario, weather_year)] = annual_load_mwh * spot_own + settlement

vppa_cost_matrix = pd.DataFrame(vppa_columns)
vppa_cost_matrix.columns.names = ["price_scenario", "weather_year"]
vppa_cost_matrix.index.name = "contract_year"
vppa_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.767057e+06  1.774301e+06  1.761999e+06  1.769014e+06   
2027                 1.742642e+06  1.749996e+06  1.738323e+06  1.745109e+06   
2028                 1.718227e+06  1.725692e+06  1.714648e+06  1.721204e+06   
2029                 1.693812e+06  1.701387e+06  1.690972e+06  1.697299e+06   
2030                 1.669396e+06  1.677082e+06  1.667296e+06  1.673394e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.734897e+06  1.687642e+06  1.759600e+06  1.744869e+06   
2027            1.712990e+06  1.671148e+06  1.735451e+06  1.721842e+06   
2028            1.691083e+06  1.654654e+06  1.711301e+06  1.698815e+06   
2029            1.669177e+06  1.638159e+06  1.687151e+06  1.675789e+06   
2030            1.647270e+06  1.621665e+06  1.663001e+06  1.652762e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.751147e+06  1.752613e+06  ...  1.807354e+06  1.828984e+06   
2027            1.728405e+06  1.729695e+06  ...  1.813989e+06  1.836369e+06   
2028            1.705663e+06  1.706777e+06  ...  1.820624e+06  1.843755e+06   
2029            1.682922e+06  1.683859e+06  ...  1.827260e+06  1.851140e+06   
2030            1.660180e+06  1.660941e+06  ...  1.833895e+06  1.858526e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.786454e+06  1.815119e+06  1.804092e+06  1.768398e+06   
2027            1.792884e+06  1.822372e+06  1.810836e+06  1.774559e+06   
2028            1.799313e+06  1.829624e+06  1.817580e+06  1.780721e+06   
2029            1.805743e+06  1.836876e+06  1.824325e+06  1.786882e+06   
2030            1.812173e+06  1.844129e+06  1.831069e+06  1.793043e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.859030e+06  1.781493e+06  1.843056e+06  1.849572e+06  
2027            1.867100e+06  1.787949e+06  1.850869e+06  1.857462e+06  
2028            1.875171e+06  1.794405e+06  1.858683e+06  1.865352e+06  
2029            1.883241e+06  1.800861e+06  1.866497e+06  1.873242e+06  
2030            1.891311e+06  1.807317e+06  1.874310e+06  1.881131e+06  

[5 rows x 99 columns]